In [1]:
print('Week 3 start')

Week 3 start


In [2]:
import pandas as pd
df = pd.read_csv('credit_risk_cleaned_v2.csv')

In [3]:
print(df['loan_grade'].unique())

<StringArray>
['B', 'C', 'A', 'D', 'E', 'F', 'G']
Length: 7, dtype: str


In [4]:
# encoding for ordered data 

grade_mapping ={'A':0,'B':1,'C':2,'D':3,'E':4,'F':5,'G':6}
df['loan_grade_encoded'] = df['loan_grade'].map(grade_mapping)
print(df['loan_grade_encoded'].unique())

[1 2 0 3 4 5 6]


In [5]:
print(df[['person_home_ownership','loan_intent']].head())

  person_home_ownership loan_intent
0                   OWN   EDUCATION
1              MORTGAGE     MEDICAL
2                  RENT     MEDICAL
3                  RENT     MEDICAL
4                   OWN     VENTURE


In [6]:
df_encoded = pd.get_dummies(df, columns=['person_home_ownership','loan_intent'])

In [7]:
print(df_encoded.columns)

Index(['Unnamed: 0', 'person_age', 'person_income', 'person_emp_length',
       'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_status',
       'loan_percent_income', 'cb_person_default_on_file',
       'cb_person_cred_hist_length', 'loan_to_income_ratio',
       'loan_grade_encoded', 'person_home_ownership_MORTGAGE',
       'person_home_ownership_OTHER', 'person_home_ownership_OWN',
       'person_home_ownership_RENT', 'loan_intent_DEBTCONSOLIDATION',
       'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT',
       'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE'],
      dtype='str')


In [8]:
# encodng for unordered data
df_encoded = pd.get_dummies(df, columns=['person_home_ownership','loan_intent'], drop_first=True)

In [9]:
print(df_encoded.columns)

Index(['Unnamed: 0', 'person_age', 'person_income', 'person_emp_length',
       'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_status',
       'loan_percent_income', 'cb_person_default_on_file',
       'cb_person_cred_hist_length', 'loan_to_income_ratio',
       'loan_grade_encoded', 'person_home_ownership_OTHER',
       'person_home_ownership_OWN', 'person_home_ownership_RENT',
       'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT',
       'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE'],
      dtype='str')


In [10]:
print(df_encoded['person_home_ownership_OWN'].head())

0     True
1    False
2    False
3    False
4     True
Name: person_home_ownership_OWN, dtype: bool


In [11]:
df_encoded['cb_person_default_on_file'] = df_encoded['cb_person_default_on_file'].map({'Y': 1, 'N': 0})

In [26]:
# THE TRAIN-TEST SPLIT---
# drop the original text columns and seperate our Target (loan_status)
X=df_encoded.drop(columns=['loan_status','loan_grade'])
y=df_encoded['loan_status']

In [13]:
from sklearn.model_selection import train_test_split
# Split into 80% training data and 20% Testing data
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42, stratify=y)
print(f"Training feature shape:{X_train.shape}")
print(f"Testing feature shape:{X_test.shape}")

Training feature shape:(26059, 19)
Testing feature shape:(6515, 19)


In [14]:
print(X_train)

       Unnamed: 0  person_age  person_income  person_emp_length  loan_amnt  \
27323       27329          35         133000                3.0      15000   
28733       28739          29          98850                6.0      12500   
2332         2338          23          33500                2.0       2500   
11621       11627          22          75000                6.0       9000   
32534       32541          52         163000                9.0      25000   
...           ...         ...            ...                ...        ...   
21942       21948          34         111280                3.0       6000   
19174       19180          27          77700                5.0       2400   
27468       27474          28         141996                3.0      13000   
29488       29494          43          29004                3.0       5000   
18983       18989          28          30000                4.0       4750   

       loan_int_rate  loan_percent_income  cb_person_default_on

In [15]:
print

<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [16]:
# --- THE NEW FIX: Slice the data to only 2 columns ---
# This ensures our scaler and model only expect 2 inputs, perfectly matching our FastAPI server.
X_train = X_train[['person_income', 'loan_amnt']]
X_test = X_test[['person_income', 'loan_amnt']]

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 1. Summon the Data Scaler
scaler = StandardScaler()

# 2. Scale the Training Data
# The scaler looks at the Training data, figures out the math to squash it, and applies it.
X_train_scaled = scaler.fit_transform(X_train)

# 3. Scale the Test Data
# IMPORTANT: It applies the EXACT SAME squashing math to the Test data.
X_test_scaled = scaler.transform(X_test)

# 4. Train the AI (Using the scaled data AND giving it more steps just in case!)
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

print("✅ Model trained successfully without warnings!")



✅ Model trained successfully without warnings!


In [17]:
# 5. Make Predictions (Remember to use the scaled test data!)
predictions = model.predict(X_test_scaled)
print('predictions',predictions)

predictions [0 0 0 ... 0 0 0]


In [18]:
# 1. Import the grading tools from scikit-learn
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 2. Overall Accuracy (The final percentage grade)
accuracy = accuracy_score(y_test, predictions)
print(f"🎯 Overall Accuracy: {accuracy * 100:.2f}%\n")

# 3. The Confusion Matrix (The detailed breakdown of right and wrong)
print("🧮 Confusion Matrix:")
print(confusion_matrix(y_test, predictions), "\n")

# 4. The Classification Report (The Professional Bank Report)
print("📊 Professional Classification Report:")
print(classification_report(y_test, predictions))

🎯 Overall Accuracy: 79.29%

🧮 Confusion Matrix:
[[5028   66]
 [1283  138]] 

📊 Professional Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.99      0.88      5094
           1       0.68      0.10      0.17      1421

    accuracy                           0.79      6515
   macro avg       0.74      0.54      0.53      6515
weighted avg       0.77      0.79      0.73      6515



In [ ]:
# 1. Ask the AI for the raw Probability Percentages instead of just 0 or 1
probabilities = model.predict_proba(X_test_scaled)[:, 1] # Get percentage for Class 1 (Default)

# 2. The Bank Manager's Cheat Code: Lower the threshold to 40% (0.40)
# If probability is >= 0.40, label them as 1 (Reject). Otherwise, 0 (Approve).
strict_threshold = 0.40
paranoid_predictions = (probabilities >= strict_threshold).astype(int)

# 3. Grade the AI again with the new Strict Rules
print(f"🚨 NEW RESULTS (Strict {strict_threshold * 100}% Threshold)")
print(f"Overall Accuracy: {accuracy_score(y_test, paranoid_predictions) * 100:.2f}%\n")

print("🧮 New Confusion Matrix:")
print(confusion_matrix(y_test, paranoid_predictions), "\n")

print("📊 New Classification Report:")
print(classification_report(y_test, paranoid_predictions))

🚨 NEW RESULTS (Strict 40.0% Threshold)
Overall Accuracy: 82.23%

🧮 New Confusion Matrix:
[[4891  203]
 [ 955  466]] 

📊 New Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.96      0.89      5094
           1       0.70      0.33      0.45      1421

    accuracy                           0.82      6515
   macro avg       0.77      0.64      0.67      6515
weighted avg       0.81      0.82      0.80      6515



In [25]:
import pickle

# 1. Save the trained AI Model
with open('bank_ai_model.pkl', 'wb') as file:
    pickle.dump(model, file)

# 2. Save the Scaler (Crucial! The webpage needs to squash new data exactly the same way)
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

print("✅ AI Brain and Scaler successfully saved to your folder!")

✅ AI Brain and Scaler successfully saved to your folder!
